# LELA60341: Research Methods in Computational and Corpus Linguistics Week 9

## A `LogisticRegression` class (1): prediction and training

Last week you wrote `TextEncoder`, which turns texts into sparse one-hot rows. This week you start `LogisticRegression`, a **subclass** of `TextEncoder` that can **predict** and **train**: the logistic regression you built in LELA60331 Week 7, but for texts. Next week, in Week 10, you will add **evaluation**.

```python
model = LogisticRegression(vocab_size=5000, lr=0.4, n_epochs=500)
model.train(train_texts, train_labels)
print(model.predict_proba(["the food was amazing !"]))
```

In this week's LELA60331 seminar (*Training and evaluating a text classifier*) you will build the same classifier step by step, and get the same results.

| Part | Topic |
|---|---|
| 0 | Set-up: the data, `TextEncoder`, and three helper functions |
| 1 | Designing `LogisticRegression` |
| 2 | Writing the class |
| 3 | Training on the Yelp reviews |
| 4 | Using the model |
| 5 | Several models side by side: the learning rate |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

Run the next three cells. The first downloads the reviews. The second sets up what you wrote in Week 8: `load_reviews`, the training and test split, and the `TextEncoder` class. The third defines three helper functions:

* `sparse_mat_vec(M, vector)`: a score for every row of a sparse matrix (LELA60331 Week 8, Activity 8)
* `sparse_transpose_mat_vec(M, vector, num_columns)`: a gradient for every column (LELA60331 Week 8, Activity 9)
* `sigmoid(z)`: a **safer** version of the sigmoid from LELA60331 Week 5. With 5000 features, the score `z` can be a very large positive or negative number, and `math.exp(-z)` is then too big for Python to store when `z` is a large negative number (try `math.exp(1000)`). For negative `z`, this version uses an equivalent formula that avoids the problem. It gives exactly the same values.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = line.rstrip().split("\t")
        texts.append(fields[0])
        labels.append(int(fields[1] == "positive"))
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")

random.seed(1)                          # the same split as in Week 8 and in LELA60331
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]
train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]


class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = []
        self._word_to_index = {}

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = counts.get(word, 0) + 1
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = [word for count, word in pairs[:self._vocab_size]]
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            self._word_to_index[self._type_list[j]] = j

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if word in self._word_to_index:
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return [self[j] for j in row]

print(len(train_texts), "training and", len(test_texts), "test reviews")

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += vector[j]
        result.append(total)
    return result

def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += vector[i]
    return result

---
# Part 1 · Designing `LogisticRegression`

`LogisticRegression` **inherits** everything from `TextEncoder`: its vocabulary, `tokenise`, `fit_vocabulary`, `one_hot_encode` and so on. It adds:

| Instance variable | What it holds |
|---|---|
| `_lr`, `_n_epochs`, `_seed` | the training settings: learning rate, number of epochs, and the random seed for the starting weights |
| `_weights`, `_bias` | the parameters: one weight per vocabulary word, and the bias |
| `_loss_history` | the loss after each epoch |

| Method | What it does | Based on LELA60331 |
|---|---|---|
| `predict_proba(texts)` | the probability that each text is positive | Week 7, Activity 7 (`mat_vec`), with `sparse_mat_vec` |
| `predict(texts)` | the predicted class, 1 or 0, of each text | Week 5, Activity 3 (`classify`) |
| `train(texts, labels)` | builds the vocabulary if there isn't one yet, then learns the weights and bias | Week 7, Activity 1 (the training loop), with `sparse_transpose_mat_vec` |
| `loss_history()` | the list of losses, for plotting | |
| `evaluate(texts, labels)` | accuracy, precision and recall | *next week* |
| `top_words(n)` | the words with the lowest and the highest weights | *next week* |

Two more design decisions:

* `predict_proba`, `predict` and `train` all take **texts**, not one-hot rows. Each one calls `self.one_hot_encode` itself, so the user of the class never needs to know about one-hot encoding.
* `train` builds the vocabulary only if the model doesn't have one yet (`len(self) == 0`: remember that `len` works on our objects). So by default the vocabulary comes from the training texts, but you can also call `fit_vocabulary` yourself first.

### 🔮 Predict

`LogisticRegression` will have its own `__init__`. If it calls `super().__init__(vocab_size)`, which class's `__init__` runs, and which instance variables does it create? Which instance variables must `LogisticRegression.__init__` then create itself?

---
# Part 2 · Writing the class

Most blanks are close to lines you wrote in the LELA60331 seminars (the table in Part 1 says where). Here is what each method does:

**`__init__`** calls the parent constructor (so that the vocabulary variables exist), then stores the training settings. Before training, there are no weights yet, so `_weights` is an empty list.

**`predict_proba(texts)`** one-hot encodes the texts, then calculates, for every row, the sigmoid of the row's score plus the bias: `sparse_mat_vec` with the rows and `self._weights` gives the scores.

**`predict(texts)`** turns each probability into 1 if it is above 0.5 and 0 otherwise. *Hint:* `int(p > 0.5)`.

**`average_loss(q, labels)`** is a helper used by `train`: the average cross-entropy loss from LELA60331 Week 5. When the score is very large, the sigmoid can round to exactly 1 (or 0), and then `math.log2(0)` gives an error. So we add a tiny number, `eps`, inside the log.

**`train(texts, labels)`** is the training loop from LELA60331 Week 7, Activity 1, with one weight per vocabulary word. As in Week 5, the model starts with random weights (and here a random bias too), and each pass through the loop calculates the probabilities, records the loss, calculates the errors and gradients, and updates the weights and the bias. Two things are different:

* the scores come from `sparse_mat_vec`, and the gradients for all the weights at once from `sparse_transpose_mat_vec` (then divided by the number of texts, as in LELA60331 Week 7, Activity 8)
* everything the loop needs to keep is an instance variable: `self._weights`, not `weights`

`verbose=True` makes it print the loss every 50 epochs.

### 🏋️ Exercise 1 · Practice — Complete `LogisticRegression`

Fill in the blanks 1–6. The cell won't run until every `???` is replaced, so work through them in order. The tests underneath train a model on a tiny corpus of eight made-up reviews, which takes no time at all.

In [ ]:
class LogisticRegression(TextEncoder):
    """Logistic regression for classifying texts, with one-hot word features."""

    def __init__(self, vocab_size=5000, lr=0.4, n_epochs=500, seed=2):
        ???                              # 1. the vocabulary variables
        self._lr = lr
        self._n_epochs = n_epochs
        self._seed = seed
        self._weights = []
        self._bias = 0
        self._loss_history = []

    # ---------------------------------------------------------------- prediction
    def predict_proba(self, texts):
        """The probability that each text is positive."""
        M = ???                            # 2.
        return ???   # 3.

    def predict(self, texts):
        """The predicted class (1 or 0) of each text."""
        return ??? # 4.

    # ---------------------------------------------------------------- training
    def average_loss(self, q, labels):
        eps = 0.00001          # a tiny number, so that we never take the log of 0
        losses = []
        for k in range(len(labels)):
            if labels[k] == 1:
                losses.append(-math.log2(q[k] + eps))
            else:
                losses.append(-math.log2(1 - q[k] + eps))
        return sum(losses) / len(losses)

    def train(self, texts, labels, verbose=False):
        """Learn the weights and bias from texts and their labels (1 or 0)."""
        if len(self) == 0:                       # no vocabulary yet
            self.fit_vocabulary(texts)
        M = self.one_hot_encode(texts)
        num_features = len(self)
        num_samples = len(labels)

        random.seed(self._seed)                  # random starting weights, as in LELA60331 Week 5
        self._weights = [random.random() for j in range(num_features)]
        self._bias = random.random()
        self._loss_history = []

        for epoch in range(self._n_epochs):
            q = [sigmoid(value + self._bias) for value in sparse_mat_vec(M, self._weights)]
            self._loss_history.append(self.average_loss(q, labels))
            # errors and gradients (LELA60331 Week 7, Activity 1, with sparse_transpose_mat_vec)
            errors = ???                                 # 5.
            dw = ???
            db = ???
            # the update
            self._weights = ???      # 6.
            self._bias = ???
            if verbose and epoch % 50 == 0:
                print("epoch", epoch, " loss", round(self._loss_history[-1], 3))

    def loss_history(self):
        return self._loss_history

tiny_texts = ["great food", "great service", "lovely food great staff", "lovely place",
              "awful food", "awful service", "rude staff awful place", "rude service"]
tiny_labels = [1, 1, 1, 1, 0, 0, 0, 0]

tiny_model = LogisticRegression(vocab_size=10, lr=0.5, n_epochs=200)
tiny_model.train(tiny_texts, tiny_labels)
print(len(tiny_model), round(tiny_model.loss_history()[0], 3), round(tiny_model.loss_history()[-1], 3))
print([round(p, 3) for p in tiny_model.predict_proba(["great place", "awful staff"])])
print(tiny_model.predict(tiny_texts))
# you should get: 8 1.595 0.046, then [0.99, 0.034], then [1, 1, 1, 1, 0, 0, 0, 0]

In [ ]:
# ✅ Check
check("1. still a TextEncoder, with an empty vocabulary", lambda: isinstance(tiny_model, TextEncoder) and len(LogisticRegression()) == 0)
check("2–3. predict_proba gives probabilities", lambda: all(0 < p < 1 for p in tiny_model.predict_proba(tiny_texts)))
check("2–3. a text with no vocabulary words gets sigmoid(bias)", lambda: abs(tiny_model.predict_proba([""])[0] - sigmoid(tiny_model._bias)) < 1e-12)
check("4. predict", lambda: tiny_model.predict(["great place", "awful staff"]) == [1, 0])
check("4. predict on the training texts", lambda: tiny_model.predict(tiny_texts) == tiny_labels)
check("5–6. the loss went down", lambda: len(tiny_model.loss_history()) == 200 and tiny_model.loss_history()[-1] < tiny_model.loss_history()[0])
check("5–6. final loss", lambda: abs(tiny_model.loss_history()[-1] - 0.0457) < 0.001)
check("5–6. 'great' has a positive weight and 'awful' a negative one",
      lambda: tiny_model._weights[tiny_model.index("great")] > 0 > tiny_model._weights[tiny_model.index("awful")])

(The checks look inside the object at `_weights` and `_bias`. That is fine for testing, but code that *uses* the class should stick to its methods.)

---
# Part 3 · Training on the Yelp reviews

### 🏋️ Exercise 2 · Practice — Training a review classifier

1. Make `model`, a `LogisticRegression` with a vocabulary of 5000 words, a learning rate of 0.4 and 500 epochs.
2. Build its vocabulary from **all** the reviews in `texts`, as in Week 8, *before* training. (This is also what this week's LELA60331 seminar does, so your results will match it.)
3. Train it on the training texts and labels. Use `verbose=True` to see its progress. Training takes about a minute.
4. Plot its loss history.

In [ ]:
model = ???   # 1.
???                                       # 2.
???              # 3.

plt.plot(???)                                    # 4.
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

In [ ]:
# ✅ Check
check("1. the settings", lambda: (len(model), model._lr, model._n_epochs) == (5000, 0.4, 500))
check("3. 500 epochs", lambda: len(model.loss_history()) == 500)
check("3. final loss 0.598", lambda: abs(model.loss_history()[-1] - 0.598) < 0.001)

The loss at the start is very high (about 8). Why? *Hint:* every starting weight is a random number between 0 and 1, so every word pushes the score **up**. What will the untrained model say about every review, and how wrong is that for the negative ones?

---
# Part 4 · Using the model

The point of wrapping everything in a class is that a trained model is easy to **use**.

### 🏋️ Exercise 3 · Practice — New reviews

1. `my_reviews`: write three short reviews of your own (in lower case, with spaces around punctuation, like the Yelp data). Make one clearly positive, one clearly negative and one mixed.
2. `my_probs`: the model's probability that each of your reviews is positive
3. `my_classes`: the model's predicted class for each of your reviews
4. `train_accuracy`: the proportion of the **training** reviews that the model classifies correctly. *Hint:* compare `model.predict(train_texts)` with `train_labels`, position by position, as in LELA60331 Week 7, Activity 2.

Do the probabilities make sense? Can you write a review that fools the model?

In [ ]:
my_reviews = [???,     # 1.
              ???,
              ???]
my_probs = ???                             # 2.
my_classes = ???                                 # 3.
for review, p, c in zip(my_reviews, my_probs, my_classes):
    print(round(p, 3), c, review)

train_predictions = model.predict(train_texts)
train_accuracy = ???   # 4.
print("training accuracy:", train_accuracy)          # you should get 0.85775

In [ ]:
# ✅ Check
check("1–2. three reviews and three probabilities", lambda: len(my_reviews) == 3 and len(my_probs) == 3 and all(0 <= p <= 1 for p in my_probs))
check("3. the classes match the probabilities", lambda: my_classes == [int(p > 0.5) for p in my_probs])
check("4. training accuracy", lambda: abs(train_accuracy - 0.85775) < 1e-9)

`zip(a, b, c)` goes through several lists side by side, giving one item from each on every pass.

The training accuracy tells us how well the model fits the reviews it learned from. It doesn't tell us how well it would do on **new** reviews. That is what the test set is for, and next week you will add an `evaluate` method to find out.

---
# Part 5 · Several models side by side: the learning rate

In LELA60331 Week 7 (Activity 3) you experimented with the learning rate by changing `lr` and re-running the training cell, which replaced the old model each time. With a class, each model is a separate object, so you can keep several and compare them.

### 🏋️ Exercise 4 · Practice — Comparing learning rates

1. `models`: a dict (Week 2) from each learning rate in `rates` to a new `LogisticRegression` with that learning rate and **100** epochs (to keep it quick). *Hint:* a loop that adds one entry per learning rate.
2. Build each model's vocabulary from all of `texts`, and train it on the training reviews.

The cell then plots the three loss histories on the same axes. Training takes about half a minute.

🔮 **Predict** before you run it: which learning rate will end with the lowest loss after 100 epochs?

In [ ]:
rates = [0.04, 0.4, 4]
models = {}
for lr in rates:
    models[lr] = ???    # 1.
    ???                                          # 2.
    ???

for lr in rates:
    plt.plot(models[lr].loss_history(), label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(models[lr].loss_history()[-1], 3))

In [ ]:
# ✅ Check
check("1. three models", lambda: sorted(models) == rates and all(isinstance(m, LogisticRegression) for m in models.values()))
check("1. each with its own learning rate", lambda: [models[lr]._lr for lr in rates] == rates)
check("2. each trained for 100 epochs", lambda: all(len(m.loss_history()) == 100 for m in models.values()))
check("the first model is unchanged", lambda: len(model.loss_history()) == 500)

Describe the three curves. Compare them with what you saw for the simulated data in LELA60331 Week 7, Activity 3.

---
## ✏️ Summary

You can now:

* write a subclass that extends its parent with new instance variables and methods, calling the parent's constructor with `super().__init__`
* design a class so that its user works with **texts**, and never needs to know about the one-hot rows inside it
* keep everything a method needs between calls (the weights, the bias, the loss history) in instance variables
* train a logistic regression model on 8000 reviews with a single method call, and use it to classify new texts
* keep several objects of the same class side by side, for example to compare settings

In this week's LELA60331 seminar you will build the same classifier step by step, and get the same loss. Next week, in Week 10, you will add **evaluation**: precision and recall on the test set, the most important words, and a better tokeniser.